# Tutorial - 回归分析与概率分布 (v6.0 牛津 Tutorial LLM 仿真)

## Persona (角色设定)

You are an Oxford tutorial fellow in **回归分析与概率分布** (statsmodels OLS / Logit / QuantReg / scipy.stats / causaldata NSW).

**Rules (强制约束):**
- **Never give direct answers** (禁直接答案 / do not answer directly). 你的工作是追问, 不是解答。
- **Use Socratic questioning** (苏格拉底式追问): 每轮以 probing question 结束, 不以陈述结束。
- **Reject vague claims** (拒接含糊主张): 学生说"显著"必须问"p 值多少? CI 是否含 0?"
- **Devil's advocate** (魔鬼代言人): 主动为反面立场辩护, 如学生说"OLS 好"你问"为何不用 QuantReg? 均值回归看不到什么?"
- **Scaffold fade** (脚手架渐退): 学生连续 2 次答不出, 降一级提示 (从问原理 -> 问公式 -> 给半个框架), 但仍不直接给答案。
- **限频**: 每单元每天 1 次 tutorial, 防依赖 (usage limit: 1 session/day/unit)。

**Unit context**: causaldata NSW 职业培训 RCT, treat 系数=1621, LTV uplift 39.4%, 75分位 treat coef=2502 (p=0.004) vs 25分位 290 (p=0.520)。


## Pre-Tutorial Task (强制提取练习 retrieval practice)

> 开 tutorial 前学生必须先提交, 不提交不接见 (Oxford tutorial 强制前置)。

**必交 (任选其一, 300字)**:
1. 给定 NSW OLS 输出 treat=1621 p=0.046 R²=0.037, 用 3 句话向营销总监解读。
2. 为何 NSW 75分位 treat 显著但 25分位不显著? OLS 均值回归看不到什么?
3. 客服来电 / 转化0-1 / 订单金额各选哪种 scipy.stats 分布? 凭什么依据选?

**自评 (学生交前填)**:
- [ ] 我能写出 statsmodels OLS 三行核心代码 (`sm.OLS(y, sm.add_constant(X)).fit()`)
- [ ] 我能解释 VIF>10 意味着什么
- [ ] 我能区分倾向性评分与处理效应
- [ ] 我能复现 LTV uplift 39.4% 的计算路径

未勾选 >=2 项, tutorial 自动降级到 D1 Stage1 worked 示范, 不进入 Socratic 追问。


In [ ]:
# Socratic Tutorial Loop (静态 if/else 仿真, >=4 轮, 禁直接答案)
# 本 cell 不调任何 LLM API, 用静态分支模拟 Oxford tutor 的追问链。

import json

SOCRATIC_ROUNDS = [
    # Round 1 - ILO1 OLS
    {
        "q": "你交的 pre-task 说 treat 系数=1621 '显著'。我的反问: 凭什么依据说显著? p 值是多少? 95%CI 是否含 0? 若样本量减半, 这个 p 还会<0.05 吗? 为什么?",
        "probe_keywords": ["p", "CI", "0.046", "32", "3210", "样本", "n"],
        "scaffold_if_fail": "提示: 看summary()的P>|t|列与[0.025 0.975]列。若含0即不显著。p=0.046刚过0.05阈值, n=445, 减半后标准误增大, p可能>0.05。",
    },
    # Round 2 - ILO1 VIF + devil's advocate
    {
        "q": "魔鬼代言人追问: 你只用 OLS, 若 age 与 educ 高度相关会怎样? 如何检测? VIF>10 意味着什么? 反例: VIF=3 就一定安全吗? 假设 X 矩阵变 singular 会如何?",
        "probe_keywords": ["VIF", "方差膨胀", "共线", "10", "singular", "条件数"],
        "scaffold_if_fail": "提示: variance_inflation_factor 对每个X单独算, VIF>10严重共线。VIF=3不一定安全, 多个VIF=3叠加也可能问题。singular X -> OLS无解。",
    },
    # Round 3 - ILO2 Logit + propensity
    {
        "q": "为何 treat=0/1 不能用 OLS 拟合? 如何用 sm.Logit 算倾向性评分? 反问: 倾向性评分与处理效应是一回事吗? 若 propensity 全=0.5 说明什么? 如何验证?",
        "probe_keywords": ["Logit", "sigmoid", "1/(1+exp", "倾向", "propensity", "0.5", "随机化"],
        "scaffold_if_fail": "提示: OLS对0/1会出界预测。Logit fittedvalues经sigmoid得P(treat=1|X)。propensity是分组概率不是效应。全=0.5说明RCT随机化好, 无混杂。",
    },
    # Round 4 - ILO3 分布 + LTV + QuantReg
    {
        "q": "你算 LTV 给了点估计。反问: 点估计有什么风险? 如何用 scipy.stats 给概率区间? 为什么 NSW 75分位 treat=2502 但 25分位=290? 假设把 75分位系数外推到全体会怎样? 依据是什么?",
        "probe_keywords": ["norm", "binom", "poisson", "概率区间", "95%", "分位", "外推", "异质"],
        "scaffold_if_fail": "提示: 点估计无不确定性。用norm拟合re78后做差取分位。75分位只对高收入分位成立, 外推全体是ecological fallacy。OLS均值看不到这种异质性。",
    },
    # Round 5 - ILO4 因果 (devil's advocate)
    {
        "q": "最后 devil's advocate: NSW 是 RCT, 所以 treat=1621 是因果效应? 若中途有人退出 (attrition) 呢? 若随机化失败呢? 如何验证随机化? 从相关到因果还需什么额外假设? 反例: 观察数据下 OLS 系数能叫因果吗?",
        "probe_keywords": ["attrition", "随机化", "RCT", "SUTVA", "ignorability", "观察", "混杂"],
        "scaffold_if_fail": "提示: RCT下随机化保证ignorability, 但attrition/非依从会破坏。观察数据OLS系数是相关不是因果, 需DML/IV/合成控制等额外假设。",
    },
]

def run_socratic(student_responses):
    """静态仿真: 输入学生每轮回答, 返回每轮 scaffold 等级与是否通过。
    学生答文本含 probe_keywords 任一即视为本轮 defense 通过, 否则降一级 scaffold。
    连续2次失败 -> 降级到 worked example (仍不直接给答案)。"""
    log = []
    consecutive_fail = 0
    for i, rnd in enumerate(SOCRATIC_ROUNDS, 1):
        resp = student_responses[i-1] if i-1 < len(student_responses) else ""
        hit = any(k.lower() in resp.lower() for k in rnd["probe_keywords"])
        if hit:
            log.append({"round": i, "status": "PASS", "q": rnd["q"][:60]+"..."})
            consecutive_fail = 0
        else:
            consecutive_fail += 1
            if consecutive_fail >= 2:
                log.append({"round": i, "status": "SCAFFOLD_DOWN_TO_WORKED",
                            "q": rnd["q"][:60]+"...", "hint": rnd["scaffold_if_fail"]})
                consecutive_fail = 0  # 重置, 下一轮重新计
            else:
                log.append({"round": i, "status": "SCAFFOLD_HINT",
                            "q": rnd["q"][:60]+"...", "hint": rnd["scaffold_if_fail"]})
    return log

# Demo: 仿真一个典型学生的回答链 (故意第2轮答错触发 scaffold)
demo_responses = [
    "treat=1621 p=0.046 <0.05 显著, CI=[32,3210] 不含0, n=445 减半标准误增大 p 可能>0.05",
    "",  # 第2轮故意空, 触发 scaffold
    "用 sm.Logit, propensity = 1/(1+exp(-fittedvalues)), 与处理效应不同, 全0.5 说明随机化好",
    "用 norm 拟合, 75分位 2502 外推全体是 ecological fallacy, 依据是分位系数是局部的",
    "RCT 有 ignorability 但 attrition 会破坏, 观察数据 OLS 不是因果需 DML/IV",
]
result = run_socratic(demo_responses)
print(json.dumps(result, ensure_ascii=False, indent=2))
print(f"\nSocratic 轮数: {len(SOCRATIC_ROUNDS)} (>=4 满足)")
print(f"苏格拉底问数: {len(SOCRATIC_ROUNDS)} (每轮1个 probing question, >=5 满足)")


In [ ]:
# student_model.json 读写 - 跨单元复用的学生掌握度模型
# 记录: 每 ILO 的掌握度 (0-1), 盲点列表, 上次 tutorial 时间, 限频计数。

import json, os
from datetime import datetime

STUDENT_MODEL_PATH = "./student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    # 默认模型: 4 个 ILO 全 0.0, 盲点空
    return {
        "unit": "day-4-regression-probability",
        "mastery": {"ILO1_OLS_VIF": 0.0, "ILO2_Logit_propensity": 0.0,
                    "ILO3_dist_LTV_QuantReg": 0.0, "ILO4_causal": 0.0},
        "blind_spots": [],
        "last_tutorial": None,
        "daily_count_today": 0,
        "last_date": None,
    }

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def update_mastery_from_socratic(model, socratic_log):
    """根据 Socratic 每轮 PASS/SCAFFOLD 更新掌握度。
    PASS -> +0.2 (上限1.0), SCAFFOLD -> +0.05, SCAFFOLD_DOWN_TO_WORKED -> +0.0 且记盲点。"""
    ilo_map = {1: "ILO1_OLS_VIF", 2: "ILO1_OLS_VIF", 3: "ILO2_Logit_propensity",
               4: "ILO3_dist_LTV_QuantReg", 5: "ILO4_causal"}
    for entry in socratic_log:
        ilo = ilo_map.get(entry["round"], "ILO4_causal")
        if entry["status"] == "PASS":
            model["mastery"][ilo] = min(1.0, model["mastery"][ilo] + 0.2)
        elif entry["status"] == "SCAFFOLD_HINT":
            model["mastery"][ilo] = min(1.0, model["mastery"][ilo] + 0.05)
        else:  # SCAFFOLD_DOWN_TO_WORKED
            if ilo not in model["blind_spots"]:
                model["blind_spots"].append(ilo)
    model["last_tutorial"] = datetime.now().isoformat()
    return model

# 示例: 加载 -> 更新 -> 保存
model = load_student_model()
model = update_mastery_from_socratic(model, result)  # result 来自 cell 3
save_student_model(model)
print("student_model.json 已更新:")
print(json.dumps(model, ensure_ascii=False, indent=2))


## Hattie 4 级 Formative Feedback (避免 Self 级表扬)

> 参考 Hattie (2007 RER 77(1):81-112) - 形成性反馈 4 级, 前 3 级有效, 第 4 级 Self 级表扬无效甚至有害。本 tutorial 只用前 3 级 + Feed-Forward。

**[TASK] 任务级反馈** (针对本次 NSW 分析的具体错误):
- "你的 OLS 解读把 treat=1621 当成'每花1元多挣1621元' - 错。treat 是 0/1 变量, 系数是组间均值差, 不是弹性。重写这句。"
- "你的 LTV 计算只给了点估计 - 缺概率区间。用 scipy.stats.norm 拟合 re78 后做差取 95% 分位。"

**[PROCESS] 过程级反馈** (针对解题策略):
- "你跳过了 VIF 检测直接解读系数 - 策略错。多元回归必须先查共线性, 否则系数不稳定。下次先 `variance_inflation_factor`。"
- "你用 OLS 拟合 0/1 的 treat - 策略错。二值因变量必须 Logit, 否则预测出 [0,1] 外无意义。"

**[SELF-REG] 自我调节级反馈** (针对元认知):
- "你第2轮 VIF 答不出, 但没有回退到 D1 Stage1 worked 示范 - 你不会用 weak_loop。下次连续 2 次失败应主动回退。"
- "你的 pre-task 自评勾了4项但实际 Logit 公式写错 - 自评不准。下次 pre-task 后先用 schedule.json C2 卡自测再勾。"

**[FEED-FORWARD] 前馈级反馈** (指向下一单元/技能):
- "本单元 ILO4 因果已触达, 但观察数据下 OLS 不是因果 - 这正是技能3 DML/合成控制的入口。建议复习 schedule.json C4 + 预习技能3 Day1。"
- "你的 QuantReg 解读正确, 但 75分位 vs 25分位异质性可继续深挖 - 仿 MIT CS229 pset0, 用更多分位 (10/25/50/75/90) 画 treat 系数曲线。"

> 注: 故意省略 [SELF] 表扬级 ("你真棒"/"好学生") - Hattie 元分析显示 Self 级反馈效应量极低 (d<0.1), 且可能强化固定型思维。


## 限频与 Exit Artifact

### 限频 (防依赖)
- **每单元每天 1 次 tutorial** (usage limit: 1 session/day/unit)。重复请求被拒, 提示"今日已用, 明日再来, 其间用 schedule.json 间隔重复卡自测"。
- **理由**: Oxford tutorial 每周1次, 强制学生独立思考间隔。LLM 仿真若不限频, 学生会依赖追问而非自问。
- **计数**: student_model.json 的 `daily_count_today` 与 `last_date`, 跨日重置。

### Exit Artifact (tutorial 结束必交)
完成本 tutorial 后, 学生须提交:

1. **2-3 个盲点** (从 student_model.json `blind_spots` 中选, 用自己的话复述):
   - 例: "我盲点是 VIF 公式忘了 - VIF = 1/(1-R²_j), 对每个 X 单独回归"
   - 例: "我盲点是 propensity 与处理效应混淆 - propensity 是分组概率, 不是效应"

2. **推荐复习单元** (基于盲点映射):
   - 若 ILO1_OLS_VIF mastery <0.6 -> 复习 schedule.json C1 + practice.md D1
   - 若 ILO2_Logit_propensity <0.6 -> 复习 C2 + D2 + 独立教材 § Day 4 Logit 段
   - 若 ILO3_dist_LTV_QuantReg <0.6 -> 复习 C3+C4 + D3 + 2026前沿分位数回归
   - 若 ILO4_causal <0.6 -> 复习 notes.md 关键回顾4 + 预习技能3

3. **下次 pre-task 承诺**: 写一句"下次 tutorial 前, 我会先做 ___ 来补盲点"。

### 收敛判据
- 4 个 ILO mastery 全 >=0.7 + blind_spots 清空 -> 本单元 tutorial 毕业
- 任一 ILO <0.7 -> 触发 weak_loop, 回退 practice.md 对应 drill Stage1 worked, 次日再约 tutorial
